# WT-01 – Winterthurer Gebäudedatensatz für die Modellanwendung vorbereiten

Dieses Notebook bereitet den gesamten Winterthurer Gebäudedatensatz so auf, dass er später konsistent für die automatisierte Bildabfrage und die Anwendung des ausgewählten WWR-Modells verwendet werden kann.

**Ziele des Notebooks**

- benötigte Pflichtspalten definieren und prüfen
- den Datensatz auf die benötigten Spalten reduzieren
- Adressen, Gebäude-IDs und Baujahrwerte vereinheitlichen
- Koordinaten initialisieren und bei Bedarf über die geo.admin.ch API ergänzen
- fehlende oder ungeeignete Kerndaten systematisch markieren
- einen adressbasierten Datensatz für die spätere Street-View-Bildabfrage vorbereiten
- eine gebäudebasierte Übersicht ableiten, falls mehrere Adresszeilen pro EGID vorhanden sind
- einen Qualitätsbericht zur Datenaufbereitung erzeugen
- den vorbereiteten Winterthurer Datensatz als Excel-Datei exportieren

Das Notebook bereitet nicht nur eine Stichprobe vor, sondern den gesamten verfügbaren Winterthurer Datensatz. Gebäude werden daher nicht ausgewählt, sondern anhand der vorhandenen Kerndaten klassifiziert und mit Statuswerten versehen.

## 1. Setup und Pfade

Die Pfade sind so definiert, dass das Notebook lokal im Projektordner ausgeführt werden kann. Die Excel-Datei sollte im Ordner `Data` liegen. Falls das Notebook in einer temporären Ausführungsumgebung getestet wird, wird zusätzlich der Pfad `/mnt/data` berücksichtigt.

In [51]:
# ============================================================
# SETUP
# ============================================================

# Falls diese Pakete in deiner Umgebung fehlen, einmalig ausführen:
# %pip install pandas openpyxl requests numpy

from pathlib import Path
from datetime import datetime
import time
import re

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

# Projektpfade
DATA_DIR = Path("Data")
OUTPUT_DIR = DATA_DIR / "WT01_modellanwendung"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Eingabedatei: lokal im Repo
INPUT_FILE = DATA_DIR / "00_Adressen_Winti_Dep_Arch.xlsx"

SHEET_NAME = "Gebäude Screening"

# Optional: Geocoding aktivieren, falls Koordinaten ergänzt werden sollen.
# Die geo.admin.ch API benötigt keinen API-Key. Trotzdem sollte die Abfrage bewusst aktiviert werden.
RUN_GEOCODING = True
GEOCODING_SLEEP_SECONDS = 0.05

# Optional: Resultate aus OPT-06 einlesen, falls bereits vorhanden.
OPT06_RESULTS_FILE = OUTPUT_DIR / "streetview_results.csv"

print("Input-Datei:", INPUT_FILE)
print("Output-Ordner:", OUTPUT_DIR)

Input-Datei: Data\00_Adressen_Winti_Dep_Arch.xlsx
Output-Ordner: Data\WT01_modellanwendung


## 2. Datensatz einlesen und Spaltenüberblick prüfen

Hier wird der originale Winterthurer Gebäudedatensatz eingelesen. Zusätzlich werden die Spaltennamen bereinigt und bei Bedarf eindeutig gemacht, damit spätere Verarbeitungsschritte stabiler funktionieren.

In [52]:
# ============================================================
# DATEN EINLESEN
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Eingabedatei nicht gefunden: {INPUT_FILE}")

def make_unique_columns(columns):
    """Macht Spaltennamen eindeutig, falls in der Excel-Datei Duplikate vorkommen."""
    seen = {}
    unique = []
    for col in columns:
        base = str(col).strip()
        if base not in seen:
            seen[base] = 0
            unique.append(base)
        else:
            seen[base] += 1
            unique.append(f"{base}_{seen[base] + 1}")
    return unique

# Rohdaten einlesen
df_raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME).copy()
original_columns = df_raw.columns.astype(str).str.strip().tolist()
df_raw.columns = make_unique_columns(original_columns)

# Dokumentieren, falls Spaltennamen mehrfach vorkamen
duplicate_columns = pd.Series(original_columns).value_counts()
duplicate_columns = duplicate_columns[duplicate_columns > 1]

if not duplicate_columns.empty:
    print("Hinweis: Es wurden doppelte Spaltennamen gefunden und eindeutig umbenannt:")
    display(duplicate_columns.rename_axis("original_column").reset_index(name="count"))

print("Shape Rohdaten:", df_raw.shape)
print("Anzahl Spalten:", len(df_raw.columns))
display(df_raw.head())

# Spaltenübersicht
columns_overview = pd.DataFrame({
    "column": list(df_raw.columns),
    "dtype": [str(df_raw.iloc[:, i].dtype) for i in range(df_raw.shape[1])],
    "missing_values": [df_raw.iloc[:, i].isna().sum() for i in range(df_raw.shape[1])],
    "missing_share": [df_raw.iloc[:, i].isna().mean() for i in range(df_raw.shape[1])],
    "nunique": [df_raw.iloc[:, i].nunique(dropna=False) for i in range(df_raw.shape[1])],
})

display(columns_overview)

Hinweis: Es wurden doppelte Spaltennamen gefunden und eindeutig umbenannt:


,original_column,count
0,Fläche,2


Shape Rohdaten: (846, 55)
Anzahl Spalten: 55


,EGID,GEB_GEBID,GSW_STATUS,STRASSENNAME,HAUSNR,HAUSNRZUSATZ,PLZ4,ORT,STADTKREIS,HAUPTNUTZUNG,NUTZUNG,GS_EIGENTUMSKATEGORIE,GS_EIGENTUMSKAT_ZUSATZ,BAUJAHR,Spalte2,Reale Baujahr,Spalte3,Link Google,Spalte4,Schadstoffen,Tragwerk Fassade6,Fassade Dämmung,zus. Eigenschaften 1,zus. Eigenschaften 2,Fassade Bekleidung,Konstruktion Decke,Bodenaufbau,Konstruktion Dach,Dach Bekleidung,Photovoltaik,PV Fläche,Fenster,Fensteranzahl,Dämmungsfläche,Stahl,Stahl lm,Stahlblech,Fläche,Eternit,Fläche 7,Steinplatten,Fläche 8,Dachziegel,Fläche 9,Beton,Fläche_2,Speziell,Fläche10,Holz,Holz Lm,Fläche11,Extra,Unnamed: 52,Unnamed: 53,Unnamed: 54
0,210294075,35999,bestehend,Albert-Einstein-Strasse,1,0,8404,Winterthur,Oberwinterthur,Industrie und Gerwerbe,Gewerbehaus,Tiefbau,APK / ueberkommunale Strassen,2024,NaN,NaN,NaN,NaN,NaN,NEIN,Punktuell Holz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ab 1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,191993323,36666,bestehend,Schützenwiesenweg,8,0,8400,Winterthur,Winterthur-Stadt,Verwaltungsgebäude und Gebäude mit öffentlichem Charakter,Klubhaus,Sportamt,übrige Sportanlagen,2024,NaN,NaN,NaN,NaN,NaN,NEIN,Massiv Beton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ab 1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,191975144,34926,im Bau,Sporrerpark,2,0,8408,Winterthur,Wülflingen,Nebengebäude und dev.Gebäude,Unterniveaugarage,Immobilien,Freihaltezonen,2024,NaN,NaN,NaN,NaN,NaN,NEIN,Massiv Beton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ab 1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,191975148,34924,im Bau,Sporrerpark,5,0,8408,Winterthur,Wülflingen,Wohngebäude,Wohnhaus,Immobilien,Freihaltezonen,2024,NaN,NaN,NaN,NaN,NaN,NEIN,Massivbau,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ab 1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,191975148,34924,im Bau,Sporrerpark,4,0,8408,Winterthur,Wülflingen,Wohngebäude,Wohnhaus,Immobilien,Freihaltezonen,2024,NaN,NaN,NaN,NaN,NaN,NEIN,Massiv Holz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ab 1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,column,dtype,missing_values,missing_share,nunique
0,EGID,int64,0,0.000000,770
1,GEB_GEBID,int64,0,0.000000,772
2,GSW_STATUS,object,0,0.000000,4
3,STRASSENNAME,object,0,0.000000,252
4,HAUSNR,int64,0,0.000000,154
5,HAUSNRZUSATZ,object,0,0.000000,8
6,PLZ4,int64,0,0.000000,9
7,ORT,object,0,0.000000,6
8,STADTKREIS,object,0,0.000000,8
9,HAUPTNUTZUNG,object,0,0.000000,9


## 3. Benötigte Spalten definieren und Datensatz reduzieren

Für die weitere Verarbeitung werden nur die Pflichtspalten beibehalten. Alle anderen Spalten werden bewusst nicht weiterverarbeitet, damit der Datensatz für die spätere Bildabfrage und Modellanwendung schlank und eindeutig bleibt.

In [53]:
# ============================================================
# SPALTENSCHEMA DEFINIEREN UND DATENSATZ REDUZIEREN
# ============================================================

REQUIRED_COLUMNS = [
    "EGID",
    "STRASSENNAME",
    "HAUSNR",
    "HAUSNRZUSATZ",
    "PLZ4",
    "ORT",
]

EXTRA_COLUMNS = [
    "BAUJAHR",
]

# Prüfen, ob alle Pflichtspalten im Rohdatensatz vorhanden sind
missing_required = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]

print("Fehlende Pflichtspalten:", missing_required)

if missing_required:
    raise ValueError(f"Es fehlen Pflichtspalten: {missing_required}")

# Optionale Zusatzspalten nur übernehmen, wenn sie vorhanden sind
available_extra_columns = [c for c in EXTRA_COLUMNS if c in df_raw.columns]

# Nur Pflichtspalten plus vorhandene Zusatzspalten behalten
columns_to_keep = REQUIRED_COLUMNS + available_extra_columns
df = df_raw[columns_to_keep].copy()

print(f"Ursprüngliche Anzahl Spalten: {df_raw.shape[1]}")
print(f"Beibehaltene Anzahl Spalten: {df.shape[1]}")
print("Beibehaltene Spalten:")
print(columns_to_keep)

display(df.head())

Fehlende Pflichtspalten: []
Ursprüngliche Anzahl Spalten: 55
Beibehaltene Anzahl Spalten: 7
Beibehaltene Spalten:
['EGID', 'STRASSENNAME', 'HAUSNR', 'HAUSNRZUSATZ', 'PLZ4', 'ORT', 'BAUJAHR']


,EGID,STRASSENNAME,HAUSNR,HAUSNRZUSATZ,PLZ4,ORT,BAUJAHR
0,210294075,Albert-Einstein-Strasse,1,0,8404,Winterthur,2024
1,191993323,Schützenwiesenweg,8,0,8400,Winterthur,2024
2,191975144,Sporrerpark,2,0,8408,Winterthur,2024
3,191975148,Sporrerpark,5,0,8408,Winterthur,2024
4,191975148,Sporrerpark,4,0,8408,Winterthur,2024


## 4. Hilfsfunktionen für Bereinigung und Statuslogik

Die folgenden Funktionen bereinigen Textwerte, nummernartige Werte und Statusbegründungen. Dadurch bleiben spätere Adress-, ID- und Statusspalten konsistent und auswertbar.

In [54]:
# ============================================================
# HILFSFUNKTIONEN
# ============================================================

def clean_text(value):
    """Bereinigt Textwerte und gibt bei fehlenden Werten einen leeren String zurück."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null", "<na>"}:
        return ""
    return re.sub(r"\s+", " ", text)


def clean_number_like_text(value):
    """Wandelt Excel-Zahlen wie 12.0 in '12' um und bereinigt leere Werte."""
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    text = clean_text(value)
    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]
    return text


def normalize_optional_house_suffix(value):
    """Bereinigt Hausnummerzusätze. Werte wie 0, 0.0 oder leer werden ignoriert."""
    text = clean_number_like_text(value)
    if text in {"", "0", "0.0"}:
        return ""
    return text


def append_reason(existing, reason):
    """Hängt einen Grund an eine semikolongetrennte Begründungsspalte an."""
    existing = clean_text(existing)
    if not existing:
        return reason
    if reason in [r.strip() for r in existing.split(";")]:
        return existing
    return f"{existing}; {reason}"


def build_address(row):
    """Erstellt eine vollständige Adresse für Geocoding und Street-View-Abfrage."""
    street = clean_text(row.get("STRASSENNAME", ""))
    house = clean_number_like_text(row.get("HAUSNR", ""))
    suffix = normalize_optional_house_suffix(row.get("HAUSNRZUSATZ", ""))
    plz = clean_number_like_text(row.get("PLZ4", ""))
    city = clean_text(row.get("ORT", ""))

    house_full = f"{house}{suffix}" if suffix else house
    street_part = f"{street} {house_full}".strip()
    city_part = f"{plz} {city}".strip()

    return f"{street_part}, {city_part}".strip(", ")


def slugify(value):
    """Erzeugt einen stabilen Dateinamen-Stub für Bild- und Exportpfade."""
    text = clean_text(value)
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue",
        "Ä": "Ae", "Ö": "Oe", "Ü": "Ue",
        "ß": "ss",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text[:120]

## 5. Basisdatensatz aufbauen

In diesem Schritt werden die Pflichtspalten bereinigt, numerische Hilfsspalten erzeugt, vollständige Adressen gebildet und doppelte EGIDs markiert.

In [55]:
# ============================================================
# BASISDATENSATZ AUFBAUEN
# ============================================================

# df wurde im vorherigen Block bereits auf die Pflichtspalten reduziert.
df = df.copy()

# ------------------------------------------------------------
# Textspalten bereinigen
# ------------------------------------------------------------
for col in ["STRASSENNAME", "ORT"]:
    df[col] = df[col].apply(clean_text)

# ------------------------------------------------------------
# Nummernartige Spalten bereinigen
# ------------------------------------------------------------
df["EGID_CLEAN"] = df["EGID"].apply(clean_number_like_text)
df["HAUSNR_CLEAN"] = df["HAUSNR"].apply(clean_number_like_text)
df["HAUSNRZUSATZ_CLEAN"] = df["HAUSNRZUSATZ"].apply(normalize_optional_house_suffix)
df["PLZ4_CLEAN"] = df["PLZ4"].apply(clean_number_like_text)

# ------------------------------------------------------------
# Baujahr optional bereinigen
# ------------------------------------------------------------

if "BAUJAHR" in df.columns:
    df["BAUJAHR_CLEAN"] = df["BAUJAHR"].apply(clean_number_like_text)
    df["BAUJAHR_NUM"] = pd.to_numeric(df["BAUJAHR_CLEAN"], errors="coerce").astype("Int64")
else:
    df["BAUJAHR"] = pd.NA
    df["BAUJAHR_CLEAN"] = ""
    df["BAUJAHR_NUM"] = pd.Series([pd.NA] * len(df), dtype="Int64")

# ------------------------------------------------------------
# Numerische Varianten erstellen
# ------------------------------------------------------------
df["EGID_NUM"] = pd.to_numeric(df["EGID_CLEAN"], errors="coerce").astype("Int64")
df["PLZ4_NUM"] = pd.to_numeric(df["PLZ4_CLEAN"], errors="coerce").astype("Int64")

# ------------------------------------------------------------
# Adresse für spätere API- und Bildabfrage erstellen
# ------------------------------------------------------------
df["address"] = df.apply(build_address, axis=1)

# Stabiler Dateiname / Identifier für spätere Bildpfade
df["address_slug"] = df.apply(
    lambda row: slugify(f"{row.get('EGID_CLEAN', '')}_{row.get('address', '')}"),
    axis=1
)

# ------------------------------------------------------------
# Dublettenlogik: dieselbe EGID kann mehrere Adresszeilen haben
# ------------------------------------------------------------
df["egid_row_count"] = df.groupby("EGID_NUM", dropna=False)["EGID_NUM"].transform("size")

# Nur gültige EGIDs als Dubletten markieren.
df["is_duplicate_egid"] = df["EGID_NUM"].notna() & (df["egid_row_count"] > 1)

print("Shape nach Basisaufbereitung:", df.shape)

display(
    df[
        [
            "EGID",
            "EGID_NUM",
            "STRASSENNAME",
            "HAUSNR_CLEAN",
            "HAUSNRZUSATZ_CLEAN",
            "PLZ4_CLEAN",
            "ORT",
            "BAUJAHR_NUM",
            "address",
            "egid_row_count",
            "is_duplicate_egid",
        ]
    ].head(10)
)

Shape nach Basisaufbereitung: (846, 19)


,EGID,EGID_NUM,STRASSENNAME,HAUSNR_CLEAN,HAUSNRZUSATZ_CLEAN,PLZ4_CLEAN,ORT,BAUJAHR_NUM,address,egid_row_count,is_duplicate_egid
0,210294075,210294075,Albert-Einstein-Strasse,1,,8404,Winterthur,2024,"Albert-Einstein-Strasse 1, 8404 Winterthur",1,False
1,191993323,191993323,Schützenwiesenweg,8,,8400,Winterthur,2024,"Schützenwiesenweg 8, 8400 Winterthur",1,False
2,191975144,191975144,Sporrerpark,2,,8408,Winterthur,2024,"Sporrerpark 2, 8408 Winterthur",1,False
3,191975148,191975148,Sporrerpark,5,,8408,Winterthur,2024,"Sporrerpark 5, 8408 Winterthur",2,True
4,191975148,191975148,Sporrerpark,4,,8408,Winterthur,2024,"Sporrerpark 4, 8408 Winterthur",2,True
5,192049948,192049948,St. Gallerstrasse,133,,8404,Winterthur,2024,"St. Gallerstrasse 133, 8404 Winterthur",1,False
6,210215067,210215067,Grüzefeldstrasse,34,,8400,Winterthur,2023,"Grüzefeldstrasse 34, 8400 Winterthur",1,False
7,210261475,210261475,Guggenbühlstrasse,140,a,8404,Winterthur,2023,"Guggenbühlstrasse 140a, 8404 Winterthur",1,False
8,191953253,191953253,Hörnlistrasse,31,,8400,Winterthur,2023,"Hörnlistrasse 31, 8400 Winterthur",1,False
9,191953634,191953634,Im Hölderli,3,,8405,Winterthur,2023,"Im Hölderli 3, 8405 Winterthur",1,False


## 6. Baujahr bereinigen und kategorisieren

Das Baujahr wird auf Plausibilität geprüft und zusätzlich in grobe Baualtersklassen eingeteilt. Ungültige oder auffällige Werte werden nicht gelöscht, sondern über eine Statusspalte markiert.

In [56]:
# ============================================================
# BAUJAHR OPTIONAL BEREINIGEN UND KATEGORISIEREN
# ============================================================

current_year = datetime.now().year

if "BAUJAHR_NUM" not in df.columns:
    if "BAUJAHR" in df.columns:
        df["BAUJAHR_NUM"] = pd.to_numeric(df["BAUJAHR"], errors="coerce").astype("Int64")
    else:
        df["BAUJAHR_NUM"] = pd.Series([pd.NA] * len(df), dtype="Int64")

invalid_baujahr_values = {0, 1000, 1200}

df["baujahr_valid"] = (
    df["BAUJAHR_NUM"].notna()
    & ~df["BAUJAHR_NUM"].isin(invalid_baujahr_values)
    & (df["BAUJAHR_NUM"] >= 1500)
    & (df["BAUJAHR_NUM"] <= current_year + 1)
)

df["BAUJAHR_KATEGORIE"] = pd.cut(
    df["BAUJAHR_NUM"].astype("float"),
    bins=[0, 1800, 1850, 1900, 1945, 1960, 1980, 2000, current_year + 1],
    labels=[
        "bis 1800",
        "1801-1850",
        "1851-1900",
        "1901-1945",
        "1946-1960",
        "1961-1980",
        "1981-2000",
        f"2001-{current_year}",
    ],
    include_lowest=True,
)

display(
    df[
        [
            "EGID",
            "BAUJAHR",
            "BAUJAHR_NUM",
            "baujahr_valid",
            "BAUJAHR_KATEGORIE",
        ]
    ].head()
)

,EGID,BAUJAHR,BAUJAHR_NUM,baujahr_valid,BAUJAHR_KATEGORIE
0,210294075,2024,2024,True,2001-2026
1,191993323,2024,2024,True,2001-2026
2,191975144,2024,2024,True,2001-2026
3,191975148,2024,2024,True,2001-2026
4,191975148,2024,2024,True,2001-2026


## 7. Koordinaten ergänzen

Der reduzierte Pflichtspalten-Datensatz enthält keine bestehenden Koordinaten. Deshalb werden Koordinatenspalten zunächst leer initialisiert. Falls `RUN_GEOCODING = True` gesetzt wird, können die Koordinaten über die Adresse mit der geo.admin.ch API ergänzt werden.

In [57]:
# ============================================================
# KOORDINATEN INITIALISIEREN
# ============================================================

df["lat"] = np.nan
df["lon"] = np.nan
df["has_coordinates"] = False

print("Keine Koordinatenspalten im reduzierten Datensatz vorhanden.")
print("Koordinaten werden optional über Geocoding anhand der Adresse ergänzt.")
print("Anzahl Zeilen mit Koordinaten:", int(df["has_coordinates"].sum()), "von", len(df))

Keine Koordinatenspalten im reduzierten Datensatz vorhanden.
Koordinaten werden optional über Geocoding anhand der Adresse ergänzt.
Anzahl Zeilen mit Koordinaten: 0 von 846


In [58]:
# ============================================================
# OPTIONALES GEOCODING ÜBER GEO.ADMIN.CH
# ============================================================

GEOADMIN_BASE = "https://api3.geo.admin.ch/rest/services/ech/SearchServer"

session = requests.Session()
session.headers.update({
    "User-Agent": "Bachelorarbeit-WWR-Winterthur-Dataset/1.0"
})

# ------------------------------------------------------------
# Geocoding-Spalten initialisieren
# ------------------------------------------------------------
# Dadurch existieren diese Spalten auch dann, wenn RUN_GEOCODING = False ist.
df["geocode_status"] = "not_run"
df["geocode_reason"] = "Geocoding was not executed."
df["lat_geocoded"] = np.nan
df["lon_geocoded"] = np.nan
df["geocode_label"] = None
df["geocode_feature_id"] = None


def geocode_address_geo_admin(address):
    """
    Geocodiert eine Adresse über geo.admin.ch und gibt ein strukturiertes Ergebnis zurück.
    """
    if not clean_text(address):
        return {
            "geocode_status": "invalid_address",
            "geocode_reason": "Address is empty.",
            "lat_geocoded": np.nan,
            "lon_geocoded": np.nan,
            "geocode_label": None,
            "geocode_feature_id": None,
            "geocode_raw": None,
        }

    params = {
        "searchText": address,
        "type": "locations",
        "origins": "address",
        "sr": 4326,
        "limit": 1,
    }

    try:
        response = session.get(GEOADMIN_BASE, params=params, timeout=30)

        if response.status_code != 200:
            return {
                "geocode_status": "http_error",
                "geocode_reason": f"HTTP status {response.status_code}",
                "lat_geocoded": np.nan,
                "lon_geocoded": np.nan,
                "geocode_label": None,
                "geocode_feature_id": None,
                "geocode_raw": response.text[:300],
            }

        data = response.json()
        results = data.get("results", [])

        if not results:
            return {
                "geocode_status": "no_result",
                "geocode_reason": "No geocoding result found.",
                "lat_geocoded": np.nan,
                "lon_geocoded": np.nan,
                "geocode_label": None,
                "geocode_feature_id": None,
                "geocode_raw": data,
            }

        best = results[0]
        attrs = best.get("attrs", {})

        lat = attrs.get("lat")
        lon = attrs.get("lon")

        # Fallback: je nach Antwortstruktur können auch y/x vorhanden sein.
        if lat is None and "y" in attrs:
            lat = attrs.get("y")
        if lon is None and "x" in attrs:
            lon = attrs.get("x")

        lat = pd.to_numeric(lat, errors="coerce")
        lon = pd.to_numeric(lon, errors="coerce")

        if pd.isna(lat) or pd.isna(lon):
            return {
                "geocode_status": "invalid_coordinates",
                "geocode_reason": "Geocoding result did not contain valid coordinates.",
                "lat_geocoded": np.nan,
                "lon_geocoded": np.nan,
                "geocode_label": attrs.get("label"),
                "geocode_feature_id": attrs.get("featureId"),
                "geocode_raw": best,
            }

        return {
            "geocode_status": "success",
            "geocode_reason": "Geocoding successful.",
            "lat_geocoded": lat,
            "lon_geocoded": lon,
            "geocode_label": attrs.get("label"),
            "geocode_feature_id": attrs.get("featureId"),
            "geocode_raw": best,
        }

    except requests.exceptions.Timeout:
        return {
            "geocode_status": "timeout",
            "geocode_reason": "Request timed out.",
            "lat_geocoded": np.nan,
            "lon_geocoded": np.nan,
            "geocode_label": None,
            "geocode_feature_id": None,
            "geocode_raw": None,
        }

    except requests.exceptions.RequestException as e:
        return {
            "geocode_status": "request_error",
            "geocode_reason": str(e),
            "lat_geocoded": np.nan,
            "lon_geocoded": np.nan,
            "geocode_label": None,
            "geocode_feature_id": None,
            "geocode_raw": None,
        }

    except ValueError as e:
        return {
            "geocode_status": "invalid_response",
            "geocode_reason": f"Response could not be parsed as JSON: {e}",
            "lat_geocoded": np.nan,
            "lon_geocoded": np.nan,
            "geocode_label": None,
            "geocode_feature_id": None,
            "geocode_raw": None,
        }


if RUN_GEOCODING:
    rows_to_geocode = df.index[
        ~df["has_coordinates"] & df["address"].ne("")
    ].tolist()

    print("Anzahl zu geocodierende Adressen:", len(rows_to_geocode))

    geocode_results = []

    for i, idx in enumerate(rows_to_geocode, start=1):
        address = df.at[idx, "address"]

        result = geocode_address_geo_admin(address)
        result["index"] = idx
        result["address"] = address

        geocode_results.append(result)

        if i % 50 == 0:
            print(f"{i}/{len(rows_to_geocode)} geocodiert")

        time.sleep(GEOCODING_SLEEP_SECONDS)

    if geocode_results:
        geocode_df = pd.DataFrame(geocode_results).set_index("index")

        cols_to_write = [
            "geocode_status",
            "geocode_reason",
            "lat_geocoded",
            "lon_geocoded",
            "geocode_label",
            "geocode_feature_id",
        ]

        for col in cols_to_write:
            df.loc[geocode_df.index, col] = geocode_df[col]

        # Koordinaten übernehmen, falls noch keine vorhanden sind
        df.loc[df["lat"].isna(), "lat"] = df.loc[df["lat"].isna(), "lat_geocoded"]
        df.loc[df["lon"].isna(), "lon"] = df.loc[df["lon"].isna(), "lon_geocoded"]

        df["has_coordinates"] = df["lat"].notna() & df["lon"].notna()

        geocode_export = OUTPUT_DIR / "WT01_geocoding_results.csv"
        geocode_df.to_csv(geocode_export, encoding="utf-8-sig")

        print("Geocoding-Ergebnisse gespeichert:", geocode_export)

    else:
        print("Keine Adressen für Geocoding vorhanden.")

else:
    print("Geocoding ist deaktiviert.")
    print("Setze RUN_GEOCODING = True, falls Koordinaten ergänzt werden sollen.")


print(
    "Anzahl Zeilen mit Koordinaten nach Geocoding-Schritt:",
    int(df["has_coordinates"].sum()),
    "von",
    len(df)
)

display(
    df[
        [
            "EGID",
            "address",
            "lat",
            "lon",
            "has_coordinates",
            "geocode_status",
            "geocode_reason",
            "geocode_label",
        ]
    ].head(10)
)

Anzahl zu geocodierende Adressen: 846
50/846 geocodiert
100/846 geocodiert
150/846 geocodiert
200/846 geocodiert
250/846 geocodiert
300/846 geocodiert
350/846 geocodiert
400/846 geocodiert
450/846 geocodiert
500/846 geocodiert
550/846 geocodiert
600/846 geocodiert
650/846 geocodiert
700/846 geocodiert
750/846 geocodiert
800/846 geocodiert
Geocoding-Ergebnisse gespeichert: Data\WT01_modellanwendung\WT01_geocoding_results.csv
Anzahl Zeilen mit Koordinaten nach Geocoding-Schritt: 846 von 846


,EGID,address,lat,lon,has_coordinates,geocode_status,geocode_reason,geocode_label
0,210294075,"Albert-Einstein-Strasse 1, 8404 Winterthur",47.515961,8.766397,True,success,Geocoding successful.,Albert-Einstein-Strasse 1 <b>8404 Winterthur</b>
1,191993323,"Schützenwiesenweg 8, 8400 Winterthur",47.500713,8.714772,True,success,Geocoding successful.,Schützenwiesenweg 8 <b>8400 Winterthur</b>
2,191975144,"Sporrerpark 2, 8408 Winterthur",47.517914,8.691791,True,success,Geocoding successful.,Sporrerpark 2 <b>8408 Winterthur</b>
3,191975148,"Sporrerpark 5, 8408 Winterthur",47.518074,8.692369,True,success,Geocoding successful.,Sporrerpark 5 <b>8408 Winterthur</b>
4,191975148,"Sporrerpark 4, 8408 Winterthur",47.518089,8.692179,True,success,Geocoding successful.,Sporrerpark 4 <b>8408 Winterthur</b>
5,192049948,"St. Gallerstrasse 133, 8404 Winterthur",47.498421,8.749280,True,success,Geocoding successful.,St. Gallerstrasse 133 <b>8404 Winterthur</b>
6,210215067,"Grüzefeldstrasse 34, 8400 Winterthur",47.494312,8.748888,True,success,Geocoding successful.,Grüzefeldstrasse 34 <b>8400 Winterthur</b>
7,210261475,"Guggenbühlstrasse 140a, 8404 Winterthur",47.518620,8.759608,True,success,Geocoding successful.,Guggenbühlstrasse 140a <b>8404 Winterthur</b>
8,191953253,"Hörnlistrasse 31, 8400 Winterthur",47.493141,8.745188,True,success,Geocoding successful.,Hörnlistrasse 31 <b>8400 Winterthur</b>
9,191953634,"Im Hölderli 3, 8405 Winterthur",47.491913,8.760794,True,success,Geocoding successful.,Im Hölderli 3 <b>8405 Winterthur</b>


## 8. Datenstatus und Ausschlussgründe ableiten

In diesem Schritt werden die Kerndaten geprüft. Problematische Zeilen werden nicht gelöscht, sondern mit Statuswerten und Begründungen markiert. Dadurch bleibt nachvollziehbar, weshalb eine Adresse für die nächste Verarbeitung bereit ist, zuerst geocodiert werden muss oder ausgeschlossen wird.

In [59]:
# ============================================================
# DATENSTATUS UND AUSSCHLUSSGRÜNDE
# ============================================================

df["data_issue_reason"] = ""

# ------------------------------------------------------------
# Pflichtinformationen prüfen
# ------------------------------------------------------------

# Fehlende oder ungültige EGID
mask_missing_egid = df["EGID_NUM"].isna()

df.loc[mask_missing_egid, "data_issue_reason"] = (
    df.loc[mask_missing_egid, "data_issue_reason"]
    .apply(lambda x: append_reason(x, "missing_egid"))
)

# Fehlende oder unvollständige Adresse
mask_missing_address = (
    df["address"].eq("")
    | df["STRASSENNAME"].eq("")
    | df["HAUSNR_CLEAN"].eq("")
    | df["PLZ4_CLEAN"].eq("")
    | df["ORT"].eq("")
)

df.loc[mask_missing_address, "data_issue_reason"] = (
    df.loc[mask_missing_address, "data_issue_reason"]
    .apply(lambda x: append_reason(x, "missing_or_invalid_address"))
)

# Fehlende Koordinaten
mask_missing_coordinates = ~df["has_coordinates"]

df.loc[mask_missing_coordinates, "data_issue_reason"] = (
    df.loc[mask_missing_coordinates, "data_issue_reason"]
    .apply(lambda x: append_reason(x, "missing_coordinates"))
)

# Fehlendes oder unplausibles Baujahr
df["data_quality_note"] = ""

mask_invalid_baujahr = ~df["baujahr_valid"]

df.loc[mask_invalid_baujahr, "data_quality_note"] = (
    df.loc[mask_invalid_baujahr, "data_quality_note"]
    .apply(lambda x: append_reason(x, "baujahr_missing_or_implausible"))
)

# ------------------------------------------------------------
# Status für Bildabfrage und spätere Modellanwendung
# ------------------------------------------------------------

conditions = [
    # Ohne EGID, Adresse oder gültiges Baujahr ist die Zeile nicht zuverlässig verwendbar.
    mask_missing_egid | mask_missing_address,

    # Kerndaten sind vorhanden, aber Koordinaten fehlen noch.
    (~mask_missing_egid)
    & (~mask_missing_address)
    & mask_missing_coordinates,

    # Kerndaten und Koordinaten sind vorhanden.
    (~mask_missing_egid)
    & (~mask_missing_address)
    & (~mask_missing_coordinates),
]

choices = [
    "exclude_missing_core_data",
    "needs_geocoding",
    "ready_for_image_request",
]

df["WT01_status"] = np.select(
    conditions,
    choices,
    default="review"
)

df["image_request_ready"] = df["WT01_status"].eq("ready_for_image_request")

df["model_application_status"] = np.where(
    df["image_request_ready"],
    "needs_image_extraction",
    df["WT01_status"]
)

# ------------------------------------------------------------
# Übersicht
# ------------------------------------------------------------

status_summary = (
    df["WT01_status"]
    .value_counts(dropna=False)
    .rename_axis("WT01_status")
    .reset_index(name="count")
)

display(status_summary)

display(
    df[
        [
            "EGID",
            "address",
            "BAUJAHR",
            "BAUJAHR_NUM",
            "baujahr_valid",
            "has_coordinates",
            "WT01_status",
            "model_application_status",
            "data_issue_reason",
        ]
    ].head(20)
)

,WT01_status,count
0,ready_for_image_request,846


,EGID,address,BAUJAHR,BAUJAHR_NUM,baujahr_valid,has_coordinates,WT01_status,model_application_status,data_issue_reason
0,210294075,"Albert-Einstein-Strasse 1, 8404 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
1,191993323,"Schützenwiesenweg 8, 8400 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
2,191975144,"Sporrerpark 2, 8408 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
3,191975148,"Sporrerpark 5, 8408 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
4,191975148,"Sporrerpark 4, 8408 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
5,192049948,"St. Gallerstrasse 133, 8404 Winterthur",2024,2024,True,True,ready_for_image_request,needs_image_extraction,
6,210215067,"Grüzefeldstrasse 34, 8400 Winterthur",2023,2023,True,True,ready_for_image_request,needs_image_extraction,
7,210261475,"Guggenbühlstrasse 140a, 8404 Winterthur",2023,2023,True,True,ready_for_image_request,needs_image_extraction,
8,191953253,"Hörnlistrasse 31, 8400 Winterthur",2023,2023,True,True,ready_for_image_request,needs_image_extraction,
9,191953634,"Im Hölderli 3, 8405 Winterthur",2023,2023,True,True,ready_for_image_request,needs_image_extraction,


## 9. Datensatz für die Bildabfrage vorbereiten

Für die spätere Street-View-Bildabfrage wird ein adressbasierter Datensatz erzeugt. Mehrere Adressen pro EGID bleiben erhalten, weil die Adresse beziehungsweise die Koordinate für die Bildabfrage relevant ist. Die spätere Auswertung kann danach wieder auf Gebäudeebene aggregiert werden.

In [60]:
# ============================================================
# ADRESSBASIERTER BILDABFRAGE-DATENSATZ
# ============================================================

# Dieser Datensatz dient als Input für die spätere Street-View-Bildabfrage.
# Er enthält nur die für die Bildabfrage und Nachvollziehbarkeit relevanten Spalten.

image_input_cols = [
    # Identifikation
    "EGID",
    "EGID_CLEAN",
    "EGID_NUM",

    # Originale und bereinigte Adressinformationen
    "STRASSENNAME",
    "HAUSNR",
    "HAUSNR_CLEAN",
    "HAUSNRZUSATZ",
    "HAUSNRZUSATZ_CLEAN",
    "PLZ4",
    "PLZ4_CLEAN",
    "ORT",

    # Baujahr
    "BAUJAHR",
    "BAUJAHR_CLEAN",
    "BAUJAHR_NUM",
    "baujahr_valid",
    "BAUJAHR_KATEGORIE",

    # Aufbereitete Adresse
    "address",
    "address_slug",

    # Koordinaten und Geocoding
    "lat",
    "lon",
    "has_coordinates",
    "geocode_status",
    "geocode_reason",
    "geocode_label",
    "geocode_feature_id",

    # WT-01 Statuslogik
    "WT01_status",
    "model_application_status",
    "data_issue_reason",
    "image_request_ready",
]

# Nur Spalten übernehmen, die tatsächlich im DataFrame existieren
image_input_cols = [c for c in image_input_cols if c in df.columns]

image_df = df[image_input_cols].copy()

# ------------------------------------------------------------
# Spalten für spätere Street-View-Bildabfrage ergänzen
# ------------------------------------------------------------

image_df["streetview_query_address"] = image_df["address"]
image_df["streetview_query_lat"] = image_df["lat"]
image_df["streetview_query_lon"] = image_df["lon"]

# Dateiname für spätere Bildspeicherung
image_df["image_filename_stub"] = image_df["address_slug"]

# Zielordner für spätere API-Bilder
image_df["expected_image_dir"] = "images_winterthur_api"

# Initialer Bildstatus
image_df["image_status"] = np.where(
    image_df["image_request_ready"],
    "not_requested",
    "not_ready"
)

# Platzhalter für spätere Bildabfrage-Ergebnisse
image_df["image_path"] = None
image_df["pano_id"] = None
image_df["metadata_status"] = None
image_df["download_status"] = None
image_df["selection_reason"] = None
image_df["quality_score"] = np.nan

print("Adressbasierter Datensatz:", image_df.shape)

display(
    image_df[
        [
            "EGID",
            "address",
            "lat",
            "lon",
            "has_coordinates",
            "BAUJAHR_NUM",
            "WT01_status",
            "image_request_ready",
            "image_status",
            "data_issue_reason",
        ]
    ].head(20)
)

Adressbasierter Datensatz: (846, 41)


,EGID,address,lat,lon,has_coordinates,BAUJAHR_NUM,WT01_status,image_request_ready,image_status,data_issue_reason
0,210294075,"Albert-Einstein-Strasse 1, 8404 Winterthur",47.515961,8.766397,True,2024,ready_for_image_request,True,not_requested,
1,191993323,"Schützenwiesenweg 8, 8400 Winterthur",47.500713,8.714772,True,2024,ready_for_image_request,True,not_requested,
2,191975144,"Sporrerpark 2, 8408 Winterthur",47.517914,8.691791,True,2024,ready_for_image_request,True,not_requested,
3,191975148,"Sporrerpark 5, 8408 Winterthur",47.518074,8.692369,True,2024,ready_for_image_request,True,not_requested,
4,191975148,"Sporrerpark 4, 8408 Winterthur",47.518089,8.692179,True,2024,ready_for_image_request,True,not_requested,
5,192049948,"St. Gallerstrasse 133, 8404 Winterthur",47.498421,8.749280,True,2024,ready_for_image_request,True,not_requested,
6,210215067,"Grüzefeldstrasse 34, 8400 Winterthur",47.494312,8.748888,True,2023,ready_for_image_request,True,not_requested,
7,210261475,"Guggenbühlstrasse 140a, 8404 Winterthur",47.518620,8.759608,True,2023,ready_for_image_request,True,not_requested,
8,191953253,"Hörnlistrasse 31, 8400 Winterthur",47.493141,8.745188,True,2023,ready_for_image_request,True,not_requested,
9,191953634,"Im Hölderli 3, 8405 Winterthur",47.491913,8.760794,True,2023,ready_for_image_request,True,not_requested,


## 10. Optional: Resultate aus OPT-06-Bildabfrage übernehmen

Falls die Bildabfrage bereits ausgeführt wurde, können die Ergebnisse hier eingelesen werden. Dadurch wird ersichtlich, welche Adressen bereits ein verwertbares Bild haben und welche nicht. Wenn keine Resultatdatei vorhanden ist, wird dieser Schritt automatisch übersprungen.

In [61]:
# ============================================================
# OPTIONALER MERGE MIT OPT-06 RESULTATEN
# ============================================================

# Erwartete Spalten in OPT06_RESULTS_FILE, falls vorhanden:
# address, status, reason, image_path, metadata_status, pano_id, heading, pitch, fov, radius, quality_score, error_detail

if OPT06_RESULTS_FILE.exists():
    opt06 = pd.read_csv(OPT06_RESULTS_FILE)
    opt06.columns = opt06.columns.astype(str).str.strip()
    print("Bildabfrage-Resultate gefunden:", OPT06_RESULTS_FILE)
    print("Shape Resultate:", opt06.shape)

    if "address" not in opt06.columns:
        raise ValueError("Die Bildabfrage-Datei enthält keine Spalte 'address'.")

    merge_cols = [c for c in [
        "address", "status", "reason", "image_path", "metadata_status", "pano_id",
        "heading", "pitch", "fov", "radius", "quality_score", "error_detail"
    ] if c in opt06.columns]

    opt06_merge = opt06[merge_cols].drop_duplicates(subset="address", keep="first").copy()
    opt06_merge = opt06_merge.rename(columns={
        "status": "image_status_opt06",
        "reason": "image_reason_opt06",
        "image_path": "image_path_opt06",
        "metadata_status": "metadata_status_opt06",
        "pano_id": "pano_id_opt06",
        "quality_score": "quality_score_opt06",
    })

    image_df = image_df.merge(opt06_merge, on="address", how="left")

    # Status aktualisieren
    image_df["image_status"] = image_df["image_status_opt06"].fillna(image_df["image_status"])
    image_df["image_path"] = image_df["image_path_opt06"].fillna(image_df["image_path"])
    image_df["pano_id"] = image_df["pano_id_opt06"].fillna(image_df["pano_id"])
    image_df["metadata_status"] = image_df["metadata_status_opt06"].fillna(image_df["metadata_status"])

    if "quality_score_opt06" in image_df.columns:
        image_df["quality_score"] = image_df["quality_score_opt06"].fillna(image_df["quality_score"])

    if "image_reason_opt06" in image_df.columns:
        image_df["selection_reason"] = image_df["image_reason_opt06"].fillna(image_df["selection_reason"])

    image_df["model_application_status"] = np.select(
        [
            image_df["image_status"].eq("success"),
            image_df["image_status"].eq("no_image_found"),
            image_df["image_status"].eq("no_suitable_candidate"),
            image_df["image_status"].eq("download_error"),
        ],
        [
            "ready_for_model",
            "no_image_available",
            "image_review_needed",
            "image_download_error",
        ],
        default=image_df["model_application_status"],
    )
else:
    print("Keine Bildabfrage-Resultatdatei gefunden. Dieser Schritt wird übersprungen:", OPT06_RESULTS_FILE)

print("Bildstatus-Übersicht:")
display(image_df["image_status"].value_counts(dropna=False).rename_axis("image_status").reset_index(name="count"))

Keine Bildabfrage-Resultatdatei gefunden. Dieser Schritt wird übersprungen: Data\WT01_modellanwendung\streetview_results.csv
Bildstatus-Übersicht:


,image_status,count
0,not_requested,846


## 11. Gebäudeebene ableiten

Für die spätere Modellanalyse kann es sinnvoll sein, pro EGID nur eine Zeile zu verwenden. Dafür wird aus dem adressbasierten Datensatz eine gebäudebasierte Tabelle erzeugt. Wenn mehrere Adressen zur gleichen EGID gehören, wird eine bevorzugte Adresse gewählt: zuerst eine mit erfolgreichem Bild, danach eine bildabfragebereite Adresse, danach eine Adresse mit möglichst wenigen Datenproblemen.

In [62]:
# ============================================================
# GEBÄUDEBASIERTER DATENSATZ
# ============================================================

# Ziel:
# Aus dem adressbasierten Datensatz wird ein gebäudebasierter Datensatz erstellt.
# Pro EGID bleibt nur eine Zeile erhalten.

priority_map = {
    # Statuswerte nach Bildabfrage
    "ready_for_model": 1,
    "needs_image_extraction": 2,
    "image_review_needed": 3,
    "no_image_available": 4,
    "image_download_error": 4,

    # aktuelle WT-01 Statuswerte
    "ready_for_image_request": 2,
    "needs_geocoding": 5,
    "review": 6,
    "exclude_missing_core_data": 9,
}

# Nur Zeilen mit gültiger EGID können gebäudebasiert zusammengeführt werden.
building_source = image_df[image_df["EGID_NUM"].notna()].copy()

# Priorität für die Auswahl der besten Adresszeile pro Gebäude
building_source["building_selection_priority"] = (
    building_source["model_application_status"]
    .map(priority_map)
    .fillna(6)
)

# Pro Gebäude die beste Zeile auswählen
building_df = (
    building_source
    .sort_values(
        ["EGID_NUM", "building_selection_priority", "address"],
        na_position="last"
    )
    .drop_duplicates(subset="EGID_NUM", keep="first")
    .reset_index(drop=True)
)

# Anzahl Adresszeilen pro Gebäude ergänzen
addr_counts = (
    image_df[image_df["EGID_NUM"].notna()]
    .groupby("EGID_NUM")
    .size()
    .rename("address_count_for_egid")
    .reset_index()
)

building_df = building_df.merge(
    addr_counts,
    on="EGID_NUM",
    how="left"
)

# Kennzeichnung, ob ein Gebäude mehrere Adresszeilen hat
building_df["has_multiple_addresses"] = building_df["address_count_for_egid"] > 1

print("Gebäudebasierter Datensatz:", building_df.shape)

display(
    building_df[
        [
            "EGID",
            "EGID_NUM",
            "address",
            "address_count_for_egid",
            "has_multiple_addresses",
            "BAUJAHR_NUM",
            "WT01_status",
            "model_application_status",
            "image_request_ready",
            "data_issue_reason",
        ]
    ].head(20)
)

Gebäudebasierter Datensatz: (770, 44)


,EGID,EGID_NUM,address,address_count_for_egid,has_multiple_addresses,BAUJAHR_NUM,WT01_status,model_application_status,image_request_ready,data_issue_reason
0,0,0,"0 0, 0 0",1,False,0,ready_for_image_request,needs_image_extraction,True,
1,1150006,1150006,"Stadthausstrasse 8b, 8400 Winterthur",1,False,1840,ready_for_image_request,needs_image_extraction,True,
2,1150007,1150007,"Stadthausstrasse 6, 8400 Winterthur",1,False,1835,ready_for_image_request,needs_image_extraction,True,
3,1150008,1150008,"Stadthausstrasse 4, 8400 Winterthur",1,False,1957,ready_for_image_request,needs_image_extraction,True,
4,1150009,1150009,"Stadthausstrasse 4a, 8400 Winterthur",1,False,1868,ready_for_image_request,needs_image_extraction,True,
5,1150014,1150014,"Lindstrasse 1, 8400 Winterthur",1,False,1863,ready_for_image_request,needs_image_extraction,True,
6,1150038,1150038,"Untertor 14, 8400 Winterthur",1,False,1680,ready_for_image_request,needs_image_extraction,True,
7,1150085,1150085,"Stadthausstrasse 37, 8400 Winterthur",1,False,1650,ready_for_image_request,needs_image_extraction,True,
8,1150131,1150131,"Oberer Graben 40, 8400 Winterthur",1,False,1972,ready_for_image_request,needs_image_extraction,True,
9,1150141,1150141,"Badgasse 8, 8400 Winterthur",1,False,1860,ready_for_image_request,needs_image_extraction,True,


## 12. Qualitätsbericht erzeugen

Der Qualitätsbericht fasst zusammen, wie viele Adressen beziehungsweise Gebäude direkt für die Bildabfrage bereit sind, wo Daten fehlen und welche Ausschluss- oder Prüffälle besonders häufig auftreten.

In [63]:
# ============================================================
# QUALITÄTSBERICHT
# ============================================================

quality_tables = {}

# Status auf adressbasierter Ebene
quality_tables["status_address_level"] = (
    image_df["model_application_status"]
    .value_counts(dropna=False)
    .rename_axis("model_application_status")
    .reset_index(name="count")
)

# Status auf gebäudebasierter Ebene
quality_tables["status_building_level"] = (
    building_df["model_application_status"]
    .value_counts(dropna=False)
    .rename_axis("model_application_status")
    .reset_index(name="count")
)

# WT-01 Status
quality_tables["wt01_status"] = (
    df["WT01_status"]
    .value_counts(dropna=False)
    .rename_axis("WT01_status")
    .reset_index(name="count")
)

# Baujahr-Validität als Zusatzinformation
# Hinweis: Ein unplausibles oder fehlendes Baujahr führt nicht zum Ausschluss.
if "baujahr_valid" in df.columns:
    quality_tables["baujahr_valid_optional"] = (
        df["baujahr_valid"]
        .value_counts(dropna=False)
        .rename_axis("baujahr_valid")
        .reset_index(name="count")
    )

# Baujahr-Kategorien als Zusatzinformation
if "BAUJAHR_KATEGORIE" in df.columns:
    quality_tables["baujahr_kategorie_optional"] = (
        df["BAUJAHR_KATEGORIE"]
        .value_counts(dropna=False)
        .rename_axis("BAUJAHR_KATEGORIE")
        .reset_index(name="count")
    )

# Koordinatenverfügbarkeit
quality_tables["coordinates"] = (
    df["has_coordinates"]
    .value_counts(dropna=False)
    .rename_axis("has_coordinates")
    .reset_index(name="count")
)

# Doppelte EGIDs
quality_tables["duplicates"] = (
    df["is_duplicate_egid"]
    .value_counts(dropna=False)
    .rename_axis("is_duplicate_egid")
    .reset_index(name="count")
)

# Mehrere Adressen pro Gebäude
if "has_multiple_addresses" in building_df.columns:
    quality_tables["multiple_addresses_per_building"] = (
        building_df["has_multiple_addresses"]
        .value_counts(dropna=False)
        .rename_axis("has_multiple_addresses")
        .reset_index(name="count")
    )

# Häufigste echte Issue-Gründe als separate Auswertung
# Diese Gründe sollten nur noch Ausschluss- oder Verarbeitungsprobleme enthalten.
issue_counts = {}

if "data_issue_reason" in df.columns:
    for reasons in df["data_issue_reason"].dropna():
        for reason in str(reasons).split(";"):
            reason = reason.strip()
            if reason:
                issue_counts[reason] = issue_counts.get(reason, 0) + 1

quality_tables["issue_reasons"] = pd.DataFrame(
    sorted(issue_counts.items(), key=lambda x: x[1], reverse=True),
    columns=["issue_reason", "count"]
)

# Optionale Qualitätsnotizen separat auswerten
# Hier kann z. B. baujahr_missing_or_implausible erscheinen.
quality_note_counts = {}

if "data_quality_note" in df.columns:
    for notes in df["data_quality_note"].dropna():
        for note in str(notes).split(";"):
            note = note.strip()
            if note:
                quality_note_counts[note] = quality_note_counts.get(note, 0) + 1

    quality_tables["quality_notes"] = pd.DataFrame(
        sorted(quality_note_counts.items(), key=lambda x: x[1], reverse=True),
        columns=["quality_note", "count"]
    )

# Ausgabe aller Qualitätsübersichten
for name, table in quality_tables.items():
    print()
    print("---", name, "---")
    display(table)


--- status_address_level ---


,model_application_status,count
0,needs_image_extraction,846



--- status_building_level ---


,model_application_status,count
0,needs_image_extraction,770



--- wt01_status ---


...

,has_multiple_addresses,count
0,False,729
1,True,41



--- issue_reasons ---


,issue_reason,count



--- quality_notes ---


,quality_note,count
0,baujahr_missing_or_implausible,33


## 13. Export speichern

Der vorbereitete Winterthurer Datensatz wird als eine Excel-Datei exportiert. Dieser Export basiert auf dem adressbasierten Datensatz und enthält alle Felder, die für die spätere Bildabfrage, Statusprüfung und Modellpipeline benötigt werden.

In [64]:
# ============================================================
# EXPORT
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

export_path = OUTPUT_DIR / f"WT01_Winterthur_vorbereiteter_Datensatz_{timestamp}.xlsx"

image_df.to_excel(export_path, index=False)

print("Vorbereiteter Winterthur-Datensatz gespeichert:")
print(export_path)

Vorbereiteter Winterthur-Datensatz gespeichert:
Data\WT01_modellanwendung\WT01_Winterthur_vorbereiteter_Datensatz_20260510_141611.xlsx


## 14. Interpretation für die Dokumentation

Diese Zelle erzeugt eine kurze textliche Zusammenfassung, die als Grundlage für die interne Dokumentation oder die Beschreibung der Datenaufbereitung verwendet werden kann.

In [65]:
# ============================================================
# KURZE TEXTLICHE ZUSAMMENFASSUNG
# ============================================================

n_rows = len(df)
n_buildings = df["EGID_NUM"].nunique(dropna=True)

n_ready_image = int((image_df["model_application_status"] == "needs_image_extraction").sum())
n_geocode = int((image_df["model_application_status"] == "needs_geocoding").sum())
n_excluded = int(image_df["model_application_status"].astype(str).str.startswith("exclude").sum())
n_review = int((image_df["model_application_status"] == "review").sum())

n_with_coordinates = int(df["has_coordinates"].sum())
n_without_coordinates = int((~df["has_coordinates"]).sum())

n_valid_baujahr = int(df["baujahr_valid"].sum())
n_invalid_baujahr = int((~df["baujahr_valid"]).sum())

summary_text = f"""
Der vorbereitete Winterthurer Gebäudedatensatz umfasst {n_rows} adressbasierte Zeilen und {n_buildings} eindeutige EGIDs.
Im Rahmen der Aufbereitung wurden die Pflichtinformationen EGID, Adresse, Hausnummer, Postleitzahl, Ort und Baujahr bereinigt und geprüft.
Aus den Adressbestandteilen wurde eine einheitliche Adressspalte für spätere Geocoding- und Street-View-Abfragen erstellt.

{n_with_coordinates} Einträge verfügen nach dem aktuellen Verarbeitungsschritt über Koordinaten, während bei {n_without_coordinates} Einträgen noch keine Koordinaten vorhanden sind.
{n_valid_baujahr} Einträge besitzen ein plausibles Baujahr. Bei {n_invalid_baujahr} Einträgen fehlt das Baujahr oder es wurde als unplausibel markiert. Diese Einträge werden dadurch nicht ausgeschlossen, da das Baujahr lediglich als Zusatzinformation verwendet wird.

{n_ready_image} adressbasierte Einträge sind für die spätere Street-View-Bildextraktion vorbereitet.
{n_geocode} Einträge benötigen zuerst eine Koordinatenergänzung.
{n_excluded} Einträge wurden aufgrund fehlender oder unplausibler Kerndaten ausgeschlossen.
{n_review} Einträge wurden als Prüffälle markiert.
""".strip()

print(summary_text)

Der vorbereitete Winterthurer Gebäudedatensatz umfasst 846 adressbasierte Zeilen und 770 eindeutige EGIDs.
Im Rahmen der Aufbereitung wurden die Pflichtinformationen EGID, Adresse, Hausnummer, Postleitzahl, Ort und Baujahr bereinigt und geprüft.
Aus den Adressbestandteilen wurde eine einheitliche Adressspalte für spätere Geocoding- und Street-View-Abfragen erstellt.

846 Einträge verfügen nach dem aktuellen Verarbeitungsschritt über Koordinaten, während bei 0 Einträgen noch keine Koordinaten vorhanden sind.
813 Einträge besitzen ein plausibles Baujahr. Bei 33 Einträgen fehlt das Baujahr oder es wurde als unplausibel markiert. Diese Einträge werden dadurch nicht ausgeschlossen, da das Baujahr lediglich als Zusatzinformation verwendet wird.

846 adressbasierte Einträge sind für die spätere Street-View-Bildextraktion vorbereitet.
0 Einträge benötigen zuerst eine Koordinatenergänzung.
0 Einträge wurden aufgrund fehlender oder unplausibler Kerndaten ausgeschlossen.
0 Einträge wurden als Prü